# Build the DiffusionFace test set

Crops a fixed sample of DiffusionFace part 1 (Chen et al., arXiv 2403.18471) with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads it as a **private** Hugging Face dataset, the test side of the `cross_diffusionface` protocol. The lab never trains on it. Unlike DiFF, the real photos are open, so this set gives an AUC on diffusion faces right away.

**Access:** part 1 is open on [Zenodo](https://zenodo.org/records/10865300) under CC BY 4.0, and the authors ask for research use only. Each folder is one tar of about 3 GB; the notebook streams every tar once and keeps only the sampled image numbers, so the archives are never stored.

**Sample:** the authors publish no test split. `diffusionface.sampled` keeps about one image number in `EVERY`, the same numbers in every folder, so each real photo sits next to the fakes made from it.

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secret `HF_TOKEN` with write access, for the upload cell.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-diffusionface-test"  # private
BUILD_DIR = "/content/build_diffusionface"
ZENODO_RECORD = "https://zenodo.org/records/10865300/files"
FOLDERS = ("Real", "ADM", "DDIM", "DDPM", "PNDM", "LDM", "MM_CelebA", "SDv15_DS0.3", "SDv15_DS0.5",
           "SDv15_DS0.7", "SDv21_DS0.3", "SDv21_DS0.5", "SDv21_DS0.7", "DiffSwap", "Inpaint")
EVERY = 20  # about 1,500 image numbers per folder

### Installation

Same as `build_dataset.ipynb`: `onnxruntime-gpu` and the ONNX backend, so RetinaFace runs on the GPU.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" gdown
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Build

Resumable, like the main build: a rerun skips every image already in `processed.jsonl`, though it streams each tar again.

In [ ]:
import requests
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, tar_members
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import diffusionface

builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector())
for folder in tqdm(FOLDERS, desc = "tars"):
    with requests.get(f"{ZENODO_RECORD}/{folder}.tar?download=1", stream = True, timeout = 120) as response:
        response.raise_for_status()
        response.raw.decode_content = True
        added = builder.add_images(tar_members(response.raw, diffusionface.parse_member,
                                               keep = lambda name: diffusionface.sampled(name, EVERY)))
    builder.flush()
    print(f"{folder}: {added} images")
print(len(builder.samples), "crops,", sum(s.is_bona_fide for s in builder.samples), "bona fide")

### Check detection outcomes

A detector that drops one folder more often than the others biases the test; compare the saved share per generator.

In [ ]:
for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:28s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

### Upload as a private dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
print(push_build(BUILD_DIR, HF_DATASET))